# 04 · Agent
Step 4: the LangGraph agent. analyze → decompose → retrieve → rerank (pass-through) → grade → answer / refuse.

Each question makes 3–4 chat calls (`gpt-4.1` for analysis, grading and the answer) plus one query embedding. Roughly a cent or two per question.

Restart the kernel after editing anything in `src/`.

In [1]:
import sys, pathlib
print(sys.executable)
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from IPython.display import Markdown, display
from finsight.config import ANALYSIS_MODEL, ANSWER_MODEL
from finsight.agent.graph import ask, get_agent
print('analysis:', ANALYSIS_MODEL, '| answer:', ANSWER_MODEL)

c:\Users\senpr\Documents\finsight-ai\.venv\Scripts\python.exe
analysis: gpt-4.1 | answer: gpt-5.4


## The graph

In [2]:
print(get_agent().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	analyze(analyze)
	decompose(decompose)
	retrieve(retrieve)
	rerank(rerank)
	grade(grade)
	answer(answer)
	refuse(refuse)
	__end__([<p>__end__</p>]):::last
	__start__ --> analyze;
	analyze -.-> decompose;
	analyze -.-> refuse;
	decompose --> retrieve;
	grade -.-> answer;
	grade -.-> refuse;
	grade -.-> retrieve;
	rerank --> grade;
	retrieve --> rerank;
	answer --> __end__;
	refuse --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Helper
Shows the answer, its citations, and the trace (what the agent extracted, searched and decided).

In [3]:
def run(question, history=None, strategy='fixed'):
    s = ask(question, history, strategy)
    a = s['analysis']
    print(f"Q: {question}")
    print(f"route: {s['route']} | in_scope: {a['in_scope']} | companies: {a['companies']} | years: {a['fiscal_years']} | type: {a['question_type']}")
    print(f"search_query: {a['search_query']!r} | sub-queries: {len(s.get('subqueries', []))} | chunks: {len(s.get('docs', []))} | grade: {s.get('grade')} | attempts: {s.get('attempts')}")
    display(Markdown(s['answer']))
    if s['citations']:
        display(pd.DataFrame(s['citations'])[['n', 'label', 'kind', 'chunk_id']])
    return s

## 1. Single lookups

In [4]:
s1 = run('What was AWS operating income in 2024?')

Q: What was AWS operating income in 2024?
route: answered | in_scope: True | companies: ['Amazon'] | years: [2024] | type: lookup
search_query: 'AWS operating income Amazon' | sub-queries: 1 | chunks: 12 | grade: {'answerable': 'yes', 'missing': ''} | attempts: 1


AWS operating income in **2024** was **$39,834 million** [1][4].

| Segment | 2024 operating income |
|---|---:|
| AWS | **$39,834 million** [1][4] |

Amazon’s MD&A also states that the increase in AWS operating income in 2024 versus 2023 was primarily due to increased sales, decreased payroll and related expenses, and lower depreciation and amortization expense from the server useful life change, partially offset by technology infrastructure spending [3].

,n,label,kind,chunk_id
0,1,Amazon annual report FY2024 — Item 7. Manageme...,table,amazon_2024-table-0004
1,3,Amazon annual report FY2024 — Item 7. Manageme...,text,amazon_2024-fixed-00084
2,4,Amazon annual report FY2024 — Note 7 — COMMITM...,table,amazon_2024-table-0031


In [5]:
s2 = run('How much revenue did Google Cloud generate in 2024?')

Q: How much revenue did Google Cloud generate in 2024?
route: answered | in_scope: True | companies: ['Alphabet'] | years: [2024] | type: lookup
search_query: 'Google Cloud revenue Alphabet' | sub-queries: 1 | chunks: 12 | grade: {'answerable': 'yes', 'missing': ''} | attempts: 1


Google Cloud generated **$43,229 million** of revenue in **2024** [1][9].

| Segment | 2024 revenue |
|---|---:|
| Google Cloud | **$43,229 million** [1][9] |

Both the segment note and the revenues-by-type table report Google Cloud 2024 revenue as **$43,229 million** [1][9].

,n,label,kind,chunk_id
0,1,Alphabet 10-K FY2024 — Notes To Consolidated F...,table,alphabet_2024-table-0056
1,9,"Alphabet 10-K FY2024 — Operations, p.37",table,alphabet_2024-table-0002


In [6]:
s3 = run('How many employees does Microsoft have?')

Q: How many employees does Microsoft have?
route: answered | in_scope: True | companies: ['Microsoft'] | years: [] | type: lookup
search_query: 'number of employees Microsoft' | sub-queries: 1 | chunks: 12 | grade: {'answerable': 'yes', 'missing': ''} | attempts: 1


As of **June 30, 2025**, Microsoft had **approximately 228,000 full-time employees**: **125,000 in the U.S.** and **103,000 internationally** [4].

| Microsoft fiscal year | Total full-time employees | U.S. | International |
|---|---:|---:|---:|
| FY2025 (as of June 30, 2025) | 228,000 | 125,000 | 103,000 [4] |
| FY2024 (as of June 30, 2024) | 228,000 | 126,000 | 102,000 [1] |
| FY2023 (as of June 30, 2023) | 221,000 | 120,000 | 101,000 [2] |

If you want, I can also break down Microsoft’s employees by function for fiscal year 2025.

,n,label,kind,chunk_id
0,1,Microsoft annual report FY2024 — Human Capital...,text,microsoft_2024-fixed-00027
1,2,Microsoft annual report FY2023 — Human Capital...,text,microsoft_2023-fixed-00028
2,4,Microsoft annual report FY2025 — Human Capital...,text,microsoft_2025-fixed-00026


## 2. Comparisons (decomposed into one sub-query per company / year)

In [7]:
s4 = run('Compare AWS and Google Cloud revenue in 2024.')

Q: Compare AWS and Google Cloud revenue in 2024.
route: answered | in_scope: True | companies: ['Amazon', 'Alphabet'] | years: [2024] | type: comparison
search_query: 'AWS revenue Amazon vs Google Cloud revenue Alphabet' | sub-queries: 2 | chunks: 12 | grade: {'answerable': 'yes', 'missing': ''} | attempts: 1


In 2024, **AWS revenue (net sales) was $107,556 million**, while **Google Cloud revenue was $43,229 million** [7][2].

| Segment | 2024 revenue | Source |
|---|---:|---|
| AWS | $107,556 million | [7] |
| Google Cloud | $43,229 million | [2] |

**Difference:** AWS exceeded Google Cloud by **$64,327 million** in 2024 [7][2].  
Calculation: **$107,556 million - $43,229 million = $64,327 million** [7][2].

Amazon also described AWS revenue as increasing from **$91B to $108B** in 2024, consistent with the table value above [1].

,n,label,kind,chunk_id
0,1,"Amazon annual report FY2024 — General, p.1",text,amazon_2024-fixed-00000
1,2,Alphabet 10-K FY2024 — Notes To Consolidated F...,table,alphabet_2024-table-0056
2,7,Amazon annual report FY2024 — Item 7. Manageme...,table,amazon_2024-table-0002


In [8]:
s5 = run('How did Microsoft Intelligent Cloud revenue change from fiscal 2023 to fiscal 2024?')

Q: How did Microsoft Intelligent Cloud revenue change from fiscal 2023 to fiscal 2024?
route: answered | in_scope: True | companies: ['Microsoft'] | years: [2023, 2024] | type: comparison
search_query: 'Intelligent Cloud revenue Microsoft' | sub-queries: 2 | chunks: 12 | grade: {'answerable': 'yes', 'missing': ''} | attempts: 1


Microsoft’s **Intelligent Cloud revenue increased by $17.5 billion, or 20%, in fiscal year 2024 compared with fiscal year 2023** [6].

| Segment | Change from fiscal 2023 to fiscal 2024 |
|---|---:|
| Intelligent Cloud revenue | **+$17.5 billion (+20%)** [6] |

Supporting detail:
- The increase was **driven by Server products and cloud services revenue**, which **increased $17.8 billion or 22%**, with **Azure and other cloud services revenue growing 30%** [6].
- This was **partly offset by Enterprise and partner services revenue decreasing $306 million or 4%** [6].

The excerpts do **not provide the absolute Intelligent Cloud revenue totals** for fiscal 2023 or fiscal 2024, only the year-over-year change [6].

,n,label,kind,chunk_id
0,6,Microsoft annual report FY2024 — Segment Resul...,text,microsoft_2024-fixed-00056


## 3. Narrative

In [9]:
s6 = run('What does Alphabet say about risks from AI regulation in 2025?')

Q: What does Alphabet say about risks from AI regulation in 2025?
route: answered | in_scope: True | companies: ['Alphabet'] | years: [2025] | type: narrative
search_query: 'AI regulation risks Alphabet' | sub-queries: 1 | chunks: 12 | grade: {'answerable': 'yes', 'missing': ''} | attempts: 1


Alphabet says **AI regulation is a growing and material risk in fiscal year 2025**, because AI laws are expanding quickly across jurisdictions, are sometimes inconsistent, and could lead to **monetary penalties, other regulatory actions, higher costs, operational changes, and legal liability** [1][3].

| AI regulation risk Alphabet highlights | What Alphabet says |
|---|---|
| Rapidly expanding regulation | Alphabet says it has seen an increase in new and evolving laws and regulations relating to **AI** in recent years, and that their scope continues to broaden in the U.S. and internationally [1]. |
| Fragmented / inconsistent rules | Alphabet says different laws on the same topic may have different or even conflicting requirements, and are not always implemented, interpreted, and enforced consistently across jurisdictions [1]. |
| Direct regulatory consequences | Alphabet says AI-focused laws and regulations could result in **“monetary penalties or other regulatory actions”** [3]. |
| Business impact | Compliance could increase Alphabet’s **cost of doing business**, make products and services **less useful**, limit certain business practices or product offerings, force changes to business models and operations, affect its competitive position, and otherwise harm its business, reputation, financial condition, and operating results [1]. |
| Legal liability from AI products | Alphabet says its AI-enabled products and services may create risks such as harmful content, inaccuracies, discrimination, IP infringement, privacy, cybersecurity, and minor protection issues, which could lead to **regulatory action** and **legal liability**, including under new and proposed AI legislation and regulations [2]. |
| Data/privacy obligations tied to AI | Alphabet says the growing use of AI has led data protection authorities to adopt new interpretations of data protection laws that impose specific obligations on processing personal data, and AI-related disclosure risks may increase scrutiny, investigations, and enforcement actions [6]. |
| Security-related regulatory exposure | Alphabet says AI features may be vulnerable to new security threats, and security incidents could lead to litigation, **regulatory fines and penalties**, and reputational harm [4]. |

A few specific 2025 examples Alphabet gives:

- The **EU AI Act** came into force on **August 1, 2024** and will generally become fully applicable after a **two-year transitional period**, with some obligations already in effect; it introduces **transparency, safety, and copyright requirements** for certain AI systems and models in the EU [3].
- In the U.S., Alphabet says **state legislatures considered more than 1,000 AI-related bills in 2025** [3].
- Alphabet specifically cites **California’s Transparency in Frontier Artificial Intelligence Act** and **New York’s Responsible AI Safety and Education Act**, saying they impose **safety and reporting obligations on developers of frontier models** [3].

In short, Alphabet’s FY2025 10-K says AI regulation risk comes from **more rules, more scrutiny, more compliance cost, and more exposure to penalties and operational constraints** [1][3][6].

,n,label,kind,chunk_id
0,1,"Alphabet 10-K FY2025 — Item 1. Business, p.8",text,alphabet_2025-fixed-00018
1,2,"Alphabet 10-K FY2025 — Item 1A. Risk Factors, ...",text,alphabet_2025-fixed-00032
2,3,"Alphabet 10-K FY2025 — Item 1A. Risk Factors, ...",text,alphabet_2025-fixed-00042
3,4,"Alphabet 10-K FY2025 — Item 1A. Risk Factors, ...",text,alphabet_2025-fixed-00036
4,6,"Alphabet 10-K FY2025 — Item 1A. Risk Factors, ...",text,alphabet_2025-fixed-00047


## 4. Refusals
Out of scope, and in scope but not in the filings.

In [10]:
s7 = run('What was Apple revenue in 2024?')

Q: What was Apple revenue in 2024?
route: out_of_scope | in_scope: False | companies: [] | years: [2024] | type: lookup
search_query: 'Apple revenue' | sub-queries: 0 | chunks: 0 | grade: None | attempts: 0


I can only answer questions about Microsoft, Amazon and Alphabet based on their annual reports / 10-Ks for fiscal years 2023–2025.

In [11]:
s8 = run('What was Amazon revenue in 2019?')

Q: What was Amazon revenue in 2019?
route: out_of_scope | in_scope: False | companies: ['Amazon'] | years: [] | type: lookup
search_query: 'Amazon revenue' | sub-queries: 0 | chunks: 0 | grade: None | attempts: 0


I can only answer questions about Microsoft, Amazon and Alphabet based on their annual reports / 10-Ks for fiscal years 2023–2025.

In [12]:
s9 = run('What is the name of the AWS CEO dog?')

Q: What is the name of the AWS CEO dog?
route: out_of_scope | in_scope: False | companies: ['Amazon'] | years: [] | type: lookup
search_query: 'AWS CEO dog name' | sub-queries: 0 | chunks: 0 | grade: None | attempts: 0


I can only answer questions about Microsoft, Amazon and Alphabet based on their annual reports / 10-Ks for fiscal years 2023–2025.

## 5. Follow-up with chat history

In [13]:
history = [{'role': 'user', 'content': s2['question']}, {'role': 'assistant', 'content': s2['answer']}]
s10 = run('And what was it in 2023?', history)

Q: And what was it in 2023?
route: answered | in_scope: True | companies: ['Alphabet'] | years: [2023] | type: lookup
search_query: 'Google Cloud revenue Alphabet' | sub-queries: 1 | chunks: 12 | grade: {'answerable': 'yes', 'missing': ''} | attempts: 1


Google Cloud generated **$33,088 million** in revenue in **2023** [6][10].

| Segment | 2023 revenue |
|---|---:|
| Google Cloud | **$33,088 million** [6][10] |

The provided excerpts include Alphabet’s **2023** figures, but do **not** include a **2024** Alphabet filing or 2024 Google Cloud revenue.

,n,label,kind,chunk_id
0,6,Alphabet 10-K FY2023 — Notes To Consolidated F...,table,alphabet_2023-table-0053
1,10,"Alphabet 10-K FY2023 — Operations, p.36",table,alphabet_2023-table-0002


## 6. Inspect the context of one answer
The excerpts the answer step saw, in order.

In [14]:
for i, d in enumerate(s4['docs'], 1):
    print(i, d.metadata['chunk_id'], d.metadata['kind'], '|', d.page_content.split(chr(10), 1)[1][:120].replace(chr(10), ' '))

1 amazon_2024-fixed-00000 text | 4 2 0 2 A N N U A L R E P O R T Dear Shareholders: 2024 was a strong year for Amazon. Our total revenue grew 11% year-ov
2 alphabet_2024-table-0056 table | Table: The following table presents revenue, profitability, and expense information about our segments (in millions): Un
3 amazon_2024-fixed-00103 text | 42 Revenue Revenue is measured based on the amount of consideration that we expect to receive, reduced by estimates for 
4 alphabet_2024-fixed-00087 text | (1) In addition to the costs included in Alphabet-level activities, hedging gains (losses) related to revenue were $236 
5 amazon_2024-fixed-00011 text | How soon? It won’t all happen in a year or two, but, it won’t take ten either. It’s moving faster than almost anything t
6 alphabet_2024-fixed-00073 text | As our business evolves, we periodically review, refine, and update our methodologies for monitoring, gathering, and cou
7 amazon_2024-table-0002 table | Table: membership fees, and certain d

## Notes / issues found
_(fill in after reviewing)_